## Getting setup

In [1]:
import torch
import torchvision

import matplotlib.pyplot as plt
from torch import nn

from going_modular.going_modular import data_setup, engine

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

### Create a helper function to set seeds

In [3]:
# Set seeds
def set_seeds(seed: int=42):

    # Set the seed for general torch operations
    torch.manual_seed(seed)
    # Set the seed for CUDA torch operations (ones that happen on the GPU)
    torch.cuda.manual_seed(seed)

---
## Get Data

In [4]:
from pathlib import Path

image_path = Path("Data/pizza_steak_sushi")

print("The folder already exists" if image_path.is_dir() else "The folder doesn't exist")

The folder already exists


---
## Create Datasets and DataLoaders

### Create DataLoaders using manually created transforms

In [5]:
from torchvision import transforms

# Setup directories
train_dir = image_path / "train"
test_dir = image_path / "test"

# Setup ImageNet normalization levels (turns all images into similar distribution as ImageNet)
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# Create transform pipeline manually
manual_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])           
print(f"Manually created transforms: {manual_transforms}")

# Create data loaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms, # use manually created transforms
    batch_size=32
)

train_dataloader, test_dataloader, class_names

Manually created transforms: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


(<torch.utils.data.dataloader.DataLoader at 0x1b103d01e80>,
 ['pizza', 'steak', 'sushi'])

### Create DataLoaders using automatically created transforms

In [6]:
# Setup dirs
train_dir = image_path / "train"
test_dir = image_path / "test"

# Setup pretrained weights (plenty of these available in torchvision.models)
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT

# Get transforms from weights (these are the transforms that were used to obtain the weights)
automatic_transforms = weights.transforms() 
print(f"Automatically created transforms: {automatic_transforms}")

# Create data loaders
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=automatic_transforms, # use automatic created transforms
    batch_size=32
)

train_dataloader, test_dataloader, class_names

Automatically created transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)


(<torch.utils.data.dataloader.DataLoader at 0x1b103d60910>,
 ['pizza', 'steak', 'sushi'])

---
## Getting a pretrained model, freezing the base layers and changing the classifier head

In [7]:
# Download the pretrained weights for EfficientNet_B0
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT # "DEFAULT" means "best weights available"

# Setup the model with the pretrained weights and send it to the target device
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

# View the output of the model
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [8]:
# Freeze all base layers by setting requires_grad attribute to False
for param in model.features.parameters():
    param.requires_grad = False
    
# Since we're creating a new layer with random weights (torch.nn.Linear), 
# let's set the seeds
set_seeds() 

# Update the classifier head to suit our problem
model.classifier = torch.nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, 
              out_features=len(class_names),
              bias=True).to(device))

---
## Train model and track results

In [9]:
# Define loss and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
import mlflow

mlflow.__version__

'3.15.1'

In [ ]:
mlflow.set_tracking_uri("sqlite:///./mlflow.db")

print("Tracking URI:")
print(mlflow.get_tracking_uri())

mlflow.set_experiment("PyTorch Experiment")

Tracking URI:
sqlite:///./mlflow.db


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1786802553369, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1786802553369, lifecycle_stage='active', name='PyTorch Experiment', tags={}, trace_location=None, workspace='default'>

In [12]:
from typing import Dict, List
from tqdm.auto import tqdm
import mlflow

from going_modular.going_modular.engine import train_step, test_step


def train(
    run_name: str,
    model: str,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: torch.nn.Module,
    epochs: int,
    device: torch.device
) -> Dict[str, List]:

    results = {
        "train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    # Start MLflow run
    with mlflow.start_run(run_name=run_name):

        print("Run ID:", mlflow.active_run().info.run_id)

        # Log parameters
        mlflow.log_params({
            "model": run_name,
            "epochs": epochs,
            "optimizer": optimizer.__class__.__name__,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "loss_function": loss_fn.__class__.__name__,
            "device": str(device)
        })

        # Training loop
        for epoch in tqdm(range(epochs)):

            train_loss, train_acc = train_step(
                model=model,
                dataloader=train_dataloader,
                loss_fn=loss_fn,
                optimizer=optimizer,
                device=device
            )

            test_loss, test_acc = test_step(
                model=model,
                dataloader=test_dataloader,
                loss_fn=loss_fn,
                device=device
            )

            print(
                f"Epoch: {epoch+1} | "
                f"train_loss: {train_loss:.4f} | "
                f"train_acc: {train_acc:.4f} | "
                f"test_loss: {test_loss:.4f} | "
                f"test_acc: {test_acc:.4f}"
            )

            # Store results
            results["train_loss"].append(train_loss)
            results["train_acc"].append(train_acc)
            results["test_loss"].append(test_loss)
            results["test_acc"].append(test_acc)

            # Log metrics
            mlflow.log_metrics({
                "train_loss": train_loss,
                "train_acc": train_acc,
                "test_loss": test_loss,
                "test_acc": test_acc
            }, step=epoch)

    return results

In [13]:
# Train model
set_seeds()
results = train(
                run_name='Effecientnet1',
                model=model,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=5,
                device=device)

Run ID: a5d6e1f951cf4f0e9cb1daa9840e4370


  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 20%|██        | 1/5 [00:47<03:08, 47.25s/it]

Epoch: 1 | train_loss: 1.0883 | train_acc: 0.4180 | test_loss: 0.8914 | test_acc: 0.6818


 40%|████      | 2/5 [01:33<02:19, 46.38s/it]

Epoch: 2 | train_loss: 0.9162 | train_acc: 0.6289 | test_loss: 0.8027 | test_acc: 0.7443


...

Epoch: 3 | train_loss: 0.8162 | train_acc: 0.7031 | test_loss: 0.6787 | test_acc: 0.9072


 80%|████████  | 4/5 [03:07<00:47, 47.05s/it]

Epoch: 4 | train_loss: 0.7460 | train_acc: 0.7305 | test_loss: 0.6744 | test_acc: 0.8040


100%|██████████| 5/5 [03:57<00:00, 47.42s/it]

Epoch: 5 | train_loss: 0.6209 | train_acc: 0.7695 | test_loss: 0.6263 | test_acc: 0.8561


In [14]:
runs = mlflow.search_runs()

runs[[
    "run_id",
    "experiment_id",
    "status",
    "start_time"
]]

,run_id,experiment_id,status,start_time
0,a5d6e1f951cf4f0e9cb1daa9840e4370,2,FINISHED,2026-08-15 17:22:26.046000+00:00
1,35aee7760e634699958dc9e5dc424a3a,2,FINISHED,2026-08-15 17:12:29.988000+00:00
2,449854a204bd4816a57cf0ffd693d1ba,2,FINISHED,2026-08-15 17:08:20.844000+00:00


In [15]:
experiment = mlflow.get_experiment_by_name("PyTorch Experiment")

print("Experiment ID:", experiment.experiment_id)
print("Experiment Name:", experiment.name)

Experiment ID: 2
Experiment Name: PyTorch Experiment


In [16]:
results

{'train_loss': [1.0882933288812637,
  0.9161567613482475,
  0.8162058144807816,
  0.7460131272673607,
  0.6208599433302879],
 'train_acc': [0.41796875, 0.62890625, 0.703125, 0.73046875, 0.76953125],
 'test_loss': [0.8914491931597391,
  0.8027182022730509,
  0.6786713202794393,
  0.6744383970896403,
  0.6262962619463602],
 'test_acc': [0.6818181818181818,
  0.7443181818181818,
  0.9071969696969697,
  0.8039772727272728,
  0.8560606060606061]}

---
## Build Another model to compare

In [17]:
# Download the pretrained weights for EfficientNet_B0
weights2 = torchvision.models.EfficientNet_B1_Weights.DEFAULT # "DEFAULT" means "best weights available"

# Setup the model with the pretrained weights and send it to the target device
model2 = torchvision.models.efficientnet_b1(weights=weights2).to(device)

# View the output of the model
model2

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [18]:
# Freeze all base layers by setting requires_grad attribute to False
for param in model2.features.parameters():
    param.requires_grad = False
    
# Since we're creating a new layer with random weights (torch.nn.Linear), 
# let's set the seeds
set_seeds() 

# Update the classifier head to suit our problem
model2.classifier = torch.nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, 
              out_features=len(class_names),
              bias=True).to(device))

In [19]:
# Define loss and optimizer
loss_fn2 = nn.CrossEntropyLoss()
optimizer2 = torch.optim.Adam(model2.parameters(), lr=0.001)

In [ ]:
# Train model
set_seeds()
results = train(
                run_name="Effecient_2v",
                model=model2,
                train_dataloader=train_dataloader,
                test_dataloader=test_dataloader,
                optimizer=optimizer2,
                loss_fn=loss_fn2,
                epochs=3,
                device=device)

Run ID: 8043847e1406407da94e02366733e28a


  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:1102: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 20%|██        | 1/5 [00:52<03:30, 52.66s/it]

Epoch: 1 | train_loss: 1.0754 | train_acc: 0.4102 | test_loss: 0.9961 | test_acc: 0.7244


 40%|████      | 2/5 [01:41<02:31, 50.64s/it]

Epoch: 2 | train_loss: 0.9795 | train_acc: 0.6797 | test_loss: 0.9209 | test_acc: 0.9072


...

Epoch: 3 | train_loss: 0.8749 | train_acc: 0.8789 | test_loss: 0.8300 | test_acc: 0.9280


 80%|████████  | 4/5 [03:13<00:47, 47.07s/it]

Epoch: 4 | train_loss: 0.8084 | train_acc: 0.7070 | test_loss: 0.7216 | test_acc: 0.9176


100%|██████████| 5/5 [03:52<00:00, 46.50s/it]

Epoch: 5 | train_loss: 0.7927 | train_acc: 0.7812 | test_loss: 0.6996 | test_acc: 0.9280
